In [1]:
import pandas as pd

import data_retrieval.order_data as order_data
from data_retrieval.order_data import get_prices_from_station
import data_retrieval.historical_data as historical_data
import data_processing.render as render
from data_processing.sde_utils import get_item_names, get_item_sizes
import os

from data_retrieval.price_differences import get_ungrouped_prices_from_station, get_volume_price_history
import settings

import nest_asyncio
from datetime import datetime, timezone

CLIENT_ID and CLIENT_SECRET found, proceeding.
loaded settings:
Settings(cache_all=False, hide_existing=False, source_hub='Jita', source_order_type='sell', dest_hub='1DQ1-A', dest_order_type='buy', days_history='7', bogus_threshold='10')


In [2]:
# Fix for Jupyter notebook
nest_asyncio.apply()
pd.set_option('display.width', 1000)

minimum_margin = 0.1 # 10%

### basically a duplicate of price_differences.py. There is probably a better way to do this.  
# give penalty score for competing sellers if settings.dest_order_type == sell
# Add sorting to output html. Maybe output as excel or csv? 
# Maybe add those variables below to settings? 

#def price_differences(settings):
print('Loading Static Data Export mappings...', end=' ')
typeIDs_to_names = get_item_names()
#typeIDs_to_sizes = get_item_sizes()
print('Done!')


Loading Static Data Export mappings... Done!


In [4]:

# Results DataFrame
summary = pd.DataFrame()


In [5]:

# Order Series
buy_prices = get_prices_from_station("Jita", "buy", False)
sell_prices = get_prices_from_station("Jita", "sell", False)
ungrouped_buy_prices = get_ungrouped_prices_from_station("Jita", "buy", False)
ungrouped_sell_prices = get_ungrouped_prices_from_station("Jita", "sell", False)

# TODO need to add perimeter citadels in a way that makes sense

#print(str(buy_prices))
#print(str(ungrouped_buy_prices))


Loading buy orders from Jita using ESI. Attempt 1...
Logged in using cached token


Loading orders: 100%|██████████| 11/11 [00:04<00:00,  2.27pages/s]


Orders from citadel 1044752365771 added to Jita orders
Saving Jita buy orders to redis cache
Got 98284 orders!
Loading sell orders from Jita using ESI. Attempt 1...
Logged in using cached token


Loading orders: 100%|██████████| 11/11 [00:01<00:00,  6.00pages/s]


Orders from citadel 1044752365771 added to Jita orders
Saving Jita sell orders to redis cache
Got 216176 orders!
Loading buy orders from Jita using ESI...
Logged in using cached token


Loading orders: 100%|██████████| 11/11 [00:01<00:00,  6.85pages/s]


Orders from citadel 1044752365771 added to Jita orders
Saving Jita buy orders to redis cache
Got 98284 orders!
       duration  is_buy_order                issued    location_id  min_volume    order_id         price    range  type_id  volume_remain  volume_total   system_id
0            90         False  2024-11-02T10:51:09Z  1044752365771           1  6906281983   72000000.00   region    19169              1             1         NaN
1            90          True  2024-09-21T11:44:22Z  1044752365771           1  6876135423 2700000000.00        1    81040              1             1         NaN
2            90          True  2024-08-24T07:22:28Z  1044752365771           1  6847889409       7601.00        1      250           1626          2000         NaN
3            90          True  2024-10-28T05:59:22Z  1044752365771           1  6902546437  351100000.00        2    19191              1             1         NaN
4            90          True  2024-11-03T00:36:04Z  1044752365771   

Loading orders: 100%|██████████| 11/11 [00:01<00:00,  6.23pages/s]


Orders from citadel 1044752365771 added to Jita orders
Saving Jita sell orders to redis cache
Got 216176 orders!
        duration  is_buy_order                issued    location_id  min_volume    order_id         price   range  type_id  volume_remain  volume_total   system_id
0             90         False  2024-11-02T10:51:09Z  1044752365771           1  6906281983   72000000.00  region    19169              1             1         NaN
1             90          True  2024-09-21T11:44:22Z  1044752365771           1  6876135423 2700000000.00       1    81040              1             1         NaN
2             90          True  2024-08-24T07:22:28Z  1044752365771           1  6847889409       7601.00       1      250           1626          2000         NaN
3             90          True  2024-10-28T05:59:22Z  1044752365771           1  6902546437  351100000.00       2    19191              1             1         NaN
4             90          True  2024-11-03T00:36:04Z  1044752365771

In [6]:
print(ungrouped_buy_prices)

       duration  is_buy_order                issued    location_id  min_volume    order_id         price    range  type_id  volume_remain  volume_total   system_id
0            90         False  2024-11-02T10:51:09Z  1044752365771           1  6906281983   72000000.00   region    19169              1             1         NaN
1            90          True  2024-09-21T11:44:22Z  1044752365771           1  6876135423 2700000000.00        1    81040              1             1         NaN
2            90          True  2024-08-24T07:22:28Z  1044752365771           1  6847889409       7601.00        1      250           1626          2000         NaN
3            90          True  2024-10-28T05:59:22Z  1044752365771           1  6902546437  351100000.00        2    19191              1             1         NaN
4            90          True  2024-11-03T00:36:04Z  1044752365771           1  6901071880        224.40        1    45500         100000        100000         NaN
...         ... 

In [7]:
ungrouped_buy_prices.loc[ungrouped_buy_prices['type_id'] == 14692]
print(ungrouped_buy_prices.loc[ungrouped_buy_prices['type_id'] == 14692])

       duration  is_buy_order                issued    location_id  min_volume    order_id        price    range  type_id  volume_remain  volume_total   system_id
6195         90          True  2024-11-01T21:01:37Z  1044752365771           1  6888410475 409300000.00   region    14692              1             1         NaN
24266        90          True  2024-10-31T20:30:35Z       60003760           1  6866237077 408000000.00   region    14692              2             2 30000142.00
26901        90          True  2024-11-02T01:35:52Z       60003760           1  6876593674 409900000.00  station    14692              1             1 30000142.00
37272        90          True  2024-11-02T08:15:29Z       60003760           1  6889460441 410000000.00  station    14692              1             1 30000142.00
45223        90          True  2024-10-27T17:09:11Z       60003760           1  6902140989 407500000.00  station    14692              1             1 30000142.00
76322        90       

In [8]:
def analyze_order_timing(df,order_type):
    """
    Analyze market orders to show how many orders were issued within different time ranges
    and calculate an activity score.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing market orders with 'issued' and 'type_id' columns
    
    Returns:
    pandas.DataFrame: Analysis of orders per time range and activity score, indexed by type_id
    """
    # Convert issued column to datetime if it's not already
    df['issued'] = pd.to_datetime(df['issued'])
    
    # Get current time in UTC
    current_time = datetime.now(timezone.utc)
    
    # Define time ranges and their scores
    time_ranges = [
        ('last_1h', pd.Timedelta(hours=1), 10),
        ('last_4h', pd.Timedelta(hours=4), 5),
        ('last_12h', pd.Timedelta(hours=12), 3),
        ('last_24h', pd.Timedelta(hours=24), 2),
        ('last_48h', pd.Timedelta(hours=48), 1)
    ]
    
    # Create a dictionary to store results
    results = {}
    
    # Calculate counts for each time range (non-overlapping)
    previous_cutoff = None
    
    for range_name, time_delta, score in time_ranges:
        cutoff_time = current_time - time_delta
        
        if previous_cutoff is None:
            # For the first range (0 to 1h)
            mask = df['issued'] >= cutoff_time
        else:
            # For subsequent ranges (e.g., 1h to 4h)
            mask = (df['issued'] >= cutoff_time) & (df['issued'] < previous_cutoff)
        
        counts = df[mask].groupby('type_id').size()
        results[range_name] = counts
        
        # Store this cutoff time for the next iteration
        previous_cutoff = cutoff_time
    
    # Combine all results into a single DataFrame
    result_df = pd.DataFrame(results)
    
    # Fill NaN values with 0 since they represent no orders in that time range
    result_df = result_df.fillna(0).astype(int)
    
    # Calculate the score
    scores = {
        'last_1h': 10,
        'last_4h': 5,
        'last_12h': 3,
        'last_24h': 2,
        'last_48h': 1
    }
    if order_type == 'buy':
        result_df['buy_competition'] = sum(result_df[col] * scores[col] for col in scores.keys())
    if order_type == 'sell':
        result_df['sell_competition'] = sum(result_df[col] * scores[col] for col in scores.keys())
    
    return result_df

In [9]:
sorted_buy_orders = analyze_order_timing(ungrouped_buy_prices,'buy')
sorted_sell_orders = analyze_order_timing(ungrouped_sell_prices,'sell')
sorted_orders = sorted_buy_orders.merge(sorted_sell_orders[['sell_competition']], how='left', left_index=True, right_index=True)


In [10]:
print(sorted_orders)
print(sorted_orders.loc[48418])

         last_1h  last_4h  last_12h  last_24h  last_48h  buy_competition  sell_competition
type_id                                                                                   
18             0        1         1         1         1               11               NaN
20             0        0         1         0         4                7             20.00
21             0        2         0         0         0               10               NaN
22             0        0         1         2         0                7              8.00
34             0        2         1         6         7               32             42.00
...          ...      ...       ...       ...       ...              ...               ...
86150          0        0         1         0         1                4              1.00
86151          0        1         1         1         0               10              8.00
86154          0        0         1         0         0                3               NaN

In [11]:
# Sell-side costs: Broker fees (~3.5%), Sales tax (~2.25%), Hauling fees (~2%).
sell_prices_taxed = sell_prices * ((1 - 0.035 - 0.0225) * (1 - 0.02))

# Series back into output DF columns
summary['Buy price'] = buy_prices
summary['Sell price'] = sell_prices_taxed
summary['Profit'] = summary['Sell price'] - summary['Buy price']
summary['Margin'] = summary['Profit'] / summary['Buy price']

# Group the remaining rows by type_id
grouped_buy_prices = ungrouped_buy_prices.groupby('type_id').agg({'price':'mean','volume_remain':'sum'})
print("grouped_buy_prices")
print(str(grouped_buy_prices))

summary = summary.merge(sorted_orders[['buy_competition','sell_competition']], how='left', on='type_id')

grouped_buy_prices.to_csv('exports/grouped_buy_prices.csv')  
# Save the total volume_remaining for each type_id
summary['Units available'] = grouped_buy_prices['volume_remain']
# Remove rows below margin treshold
summary = summary[summary['Margin'] >= minimum_margin]

#summary = summary.fillna(0).astype(int)

# Print summary DataFrame to check values
print("summary:")
print(str(summary))
# Get historical volume and price for order refinement


grouped_buy_prices
               price  volume_remain
type_id                            
18             38.74       49634052
19          12050.00          36821
20            755.28        3121213
21            882.52         163803
22           5353.75          62279
...              ...            ...
86152    39755000.00              8
86153   834109062.50             36
86154   186433333.33              8
86155     4765620.00             26
86156   361800000.00              8

[15782 rows x 2 columns]
summary:
           Buy price   Sell price       Profit  Margin  buy_competition  sell_competition  Units available
type_id                                                                                                   
19          15000.00     27700.26     12700.26    0.85              NaN               NaN            36821
21           1015.00      1616.39       601.39    0.59            10.00               NaN           163803
43            122.80       282.18       159.38    

In [12]:

volume, price = get_volume_price_history("Jita", summary.index, 7, False)

summary['Daily Volume'] = volume
summary['Avg Price'] = price


Loading market history in Jita for 8902 items...


Success!
Processing history content into dataframe...
Done!
Pulling out historical volume data for 7 days...
              2024-10-28  2024-10-29  2024-10-30  2024-10-31  2024-11-01  2024-11-02
19    volume     3322.00      587.00      555.00      803.00      104.00        6.00
21    volume     1500.00         NaN     1683.00      166.00      551.00       54.00
43    volume      259.00   262492.00    97848.00   120864.00   107453.00    95698.00
45    volume      640.00      412.00      578.00      779.00      290.00      336.00
178   volume   393910.00   127136.00   244926.00  1242559.00    69350.00    85891.00
...                  ...         ...         ...         ...         ...         ...
85754 volume        4.00        3.00       16.00         NaN         NaN        7.00
85755 volume       17.00        7.00         NaN        4.00        3.00        8.00
86143 volume       22.00       56.00       64.00       26.00        3.00        6.00
86149 volume        9.00        2.00     

In [13]:
#bogus_threshold = 100

# Remove orders where the ratio of current price to historic price is too high
#summary = summary[(summary['Sell price'] / price) < float(bogus_threshold)]

# Finally, calculate expected profit per item by multiplying unit profit by mean daily volume or number of available units, whichever is lower
summary['Total Profit'] = summary['Profit'] * summary.apply(lambda row: min(row['Units available'], row['Daily Volume']), axis=1)

summary['Daily Profit'] = summary['Profit'] * summary['Daily Volume']

# Clean up by dropping NAN rows - Usually this means no volume data or no sell-side orders.
#summary = summary.dropna()

# Sort by profit descending for output
summary = summary.sort_values(by='Daily Profit', ascending=False)

# Use item names instead of typeIDs
summary['typeID'] = summary.index
summary = summary.rename(typeIDs_to_names)
summary.to_csv('exports/summary.csv') 

print (summary)

                                                  Buy price     Sell price         Profit  Margin  buy_competition  sell_competition  Units available  Daily Volume      Avg Price    Total Profit   Daily Profit  typeID
type_id                                                                                                                                                                                                                  
Estamel's Modified Torpedo Launcher           5723000000.00 14529014500.00  8806014500.00    1.54              NaN               NaN            10009          1.33 11823750000.00  11741352666.67 11741352666.67   14683
Looking Glass Monocle Interface (right/gold) 24600000000.00 59575425000.00 34975425000.00    1.42             6.00               NaN                2          0.33 41785000000.00  11658475000.00 11658475000.00    4009
Standup L-Set Reprocessing Monitor II           25190000.00 31376390500.00 31351200500.00 1244.59              NaN              

In [14]:
test_summary = summary

test_summary['Trade score'] = test_summary['Profit'] * (test_summary['Daily Volume']/2) / ((test_summary['buy_competition']+test_summary['sell_competition'])*10000)

test_summary = test_summary.sort_values(by='Trade score', ascending=False)
print(test_summary)

                                                 Buy price     Sell price         Profit  Margin  buy_competition  sell_competition  Units available  Daily Volume     Avg Price    Total Profit  Daily Profit  typeID  Trade score
type_id                                                                                                                                                                                                                            
Amarr Titan                                   931000000.00  6080387950.00  5149387950.00    5.53             3.00              1.00                1          1.00 6048875000.00   5149387950.00 5149387950.00    3347    128734.70
Raven Blueprint                                  119200.00  1048342750.00  1048223550.00 8793.82             2.00              3.00               15          2.50 1087713295.83   2620558875.00 2620558875.00     688     52411.18
Drake Blueprint                               143000000.00   535717000.00   392717000.00

In [17]:

output_summary = test_summary[['Buy price', 'Sell price','Profit', 'Margin',
        'Daily Volume', 'Daily Profit', 
        'Trade score'
]]
print(output_summary)
# Output to a pretty table and return URL for further use
url = render.render_to_html(output_summary)

                                                        Buy price           Sell price               Profit               Margin         Daily Volume         Daily Profit          Trade score
type_id                                                                                                                                                                                        
Amarr Titan                                        931,000,000.00     6,080,387,950.00     5,149,387,950.00                 5.53                 1.00     5,149,387,950.00           128,734.70
Raven Blueprint                                        119,200.00     1,048,342,750.00     1,048,223,550.00             8,793.82                 2.50     2,620,558,875.00            52,411.18
Drake Blueprint                                    143,000,000.00       535,717,000.00       392,717,000.00                 2.75                 3.83     1,505,415,166.67            50,180.51
Women's 'RubySet TorsoRig' Body Augmenta

[15:17:52] PaIadin > <url=showinfo:35833//1044752365771>Perimeter - 0.0% Fence Tax Haven</url>
